# Qwen2.5-Coder Quantization Comparison: FP16 vs INT8 vs INT4

This notebook compares different quantization levels for Qwen2.5-Coder-7B-Instruct to help you choose the best balance between model size, speed, and code quality.

### Quantization Overview

| Format | Model Size | Memory | Speed | Quality |
|--------|------------|--------|-------|----------|
| FP16 | ~14 GB | ~16 GB | Baseline | Best |
| INT8 | ~7 GB | ~8 GB | ~1.5x faster | Very Good |
| INT4 | ~4 GB | ~5 GB | ~2x faster | Good |

### What We'll Compare

1. **Model Size**: Disk space required
2. **Inference Speed**: Tokens per second
3. **Memory Usage**: RAM/VRAM consumption
4. **Code Quality**: Evaluating generated code on multiple programming tasks

### Recommendation

For most users with **16GB RAM + 8GB GPU**:
- **INT4** is recommended for best balance
- **INT8** if you have slightly more memory and want better quality
- **FP16** only if you have 32GB+ RAM

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert Models with Different Quantization](#Convert-Models)
- [Compare Model Sizes](#Compare-Model-Sizes)
- [Benchmark Inference Speed](#Benchmark-Speed)
- [Evaluate Code Quality](#Evaluate-Quality)
- [Summary and Recommendation](#Summary)

## Prerequisites
[back to top](#Table-of-contents:)

In [ ]:
import requests
from pathlib import Path
import sys

if not Path("qwen2_5_coder_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/Qwen2.5-Coder/qwen2_5_coder_helper.py",
    )
    open("qwen2_5_coder_helper.py", "w").write(r.text)

if not Path("gradio_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/Qwen2.5-Coder/gradio_helper.py",
    )
    open("gradio_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

In [ ]:
from pip_helper import pip_install

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch==2.9.1",
    "torchvision==0.24.1",
    "torchaudio==2.9.1",
    "nncf>=2.18.0",
    "gradio>=4.19",
    "huggingface_hub",
    "matplotlib",
    "pandas",
)

pip_install(
    "transformers>=4.37.0",
    "accelerate",
)

from notebook_utils import collect_telemetry
collect_telemetry("quantization_comparison.ipynb")

## Convert Models with Different Quantization
[back to top](#Table-of-contents:)

We'll convert Qwen2.5-Coder-7B-Instruct to three formats:
1. **FP16**: Full precision (baseline)
2. **INT8**: 8-bit quantization
3. **INT4**: 4-bit quantization

In [ ]:
import ipywidgets as widgets

model_ids = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
]

model_selector = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
    style={"description_width": "initial"},
)

model_selector

In [ ]:
model_id = model_selector.value
base_model_name = model_id.split("/")[-1]

# Directories for each quantization level
fp16_dir = Path(f"{base_model_name}-FP16")
int8_dir = Path(f"{base_model_name}-INT8")
int4_dir = Path(f"{base_model_name}-INT4")

print(f"FP16 directory: {fp16_dir}")
print(f"INT8 directory: {int8_dir}")
print(f"INT4 directory: {int4_dir}")

In [ ]:
import nncf
from qwen2_5_coder_helper import convert_qwen2_5_coder_model

# Convert FP16 (baseline)
print("=" * 60)
print("Converting FP16 model...")
print("=" * 60)
convert_qwen2_5_coder_model(
    model_id=model_id,
    output_dir=fp16_dir,
    quantization_config=None,  # FP16
)

In [ ]:
# Convert INT8
print("=" * 60)
print("Converting INT8 model...")
print("=" * 60)
convert_qwen2_5_coder_model(
    model_id=model_id,
    output_dir=int8_dir,
    quantization_config={
        "mode": nncf.CompressWeightsMode.INT8_SYM,
    },
)

In [ ]:
# Convert INT4
print("=" * 60)
print("Converting INT4 model...")
print("=" * 60)
convert_qwen2_5_coder_model(
    model_id=model_id,
    output_dir=int4_dir,
    quantization_config={
        "mode": nncf.CompressWeightsMode.INT4_ASYM,
        "group_size": 128,
        "ratio": 0.8,
    },
)

## Compare Model Sizes
[back to top](#Table-of-contents:)

In [ ]:
import os
import pandas as pd

def get_dir_size(path):
    """Get total size of directory in GB."""
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total / (1024**3)  # Convert to GB

# Get sizes
fp16_size = get_dir_size(fp16_dir) if fp16_dir.exists() else 0
int8_size = get_dir_size(int8_dir) if int8_dir.exists() else 0
int4_size = get_dir_size(int4_dir) if int4_dir.exists() else 0

# Create comparison table
size_data = {
    "Format": ["FP16", "INT8", "INT4"],
    "Size (GB)": [fp16_size, int8_size, int4_size],
    "Reduction": ["Baseline", f"{(1-int8_size/fp16_size)*100:.1f}%", f"{(1-int4_size/fp16_size)*100:.1f}%"],
}

df_size = pd.DataFrame(size_data)
print("Model Size Comparison:")
print(df_size.to_string(index=False))

# Visualize
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(8, 4))
    formats = ["FP16", "INT8", "INT4"]
    sizes = [fp16_size, int8_size, int4_size]
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    
    bars = ax.bar(formats, sizes, color=colors)
    ax.set_ylabel("Size (GB)")
    ax.set_title("Model Size by Quantization Format")
    
    # Add value labels
    for bar, size in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{size:.2f} GB", ha="center", va="bottom")
    
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for visualization: pip install matplotlib")

## Benchmark Inference Speed
[back to top](#Table-of-contents:)

Measure tokens per second for each quantization format.

In [ ]:
import time
import numpy as np
from qwen2_5_coder_helper import OVQwen2_5CoderForCausalLM
from transformers import AutoConfig, AutoTokenizer

# Load tokenizer (same for all formats)
tokenizer = AutoTokenizer.from_pretrained(fp16_dir, trust_remote_code=True)
config = AutoConfig.from_pretrained(fp16_dir, trust_remote_code=True)

# Test prompt
test_prompt = "Write a Python function to calculate the Fibonacci sequence."
messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

# Benchmark parameters
num_runs = 3
max_new_tokens = 256

results = {}

for format_name, model_dir in [("FP16", fp16_dir), ("INT8", int8_dir), ("INT4", int4_dir)]:
    if not model_dir.exists():
        print(f"Skipping {format_name} - directory not found")
        continue
    
    print(f"\nBenchmarking {format_name}...")
    
    # Load model
    model = OVQwen2_5CoderForCausalLM(model_dir=model_dir, device="CPU", config=config)
    
    # Warmup run
    _ = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    
    # Benchmark runs
    times = []
    token_counts = []
    
    for i in range(num_runs):
        start_time = time.time()
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
        elapsed = time.time() - start_time
        num_tokens = output_ids.shape[1] - inputs["input_ids"].shape[1]
        
        times.append(elapsed)
        token_counts.append(num_tokens)
    
    avg_time = np.mean(times)
    avg_tokens = np.mean(token_counts)
    tokens_per_sec = avg_tokens / avg_time
    
    results[format_name] = {
        "avg_time": avg_time,
        "avg_tokens": avg_tokens,
        "tokens_per_sec": tokens_per_sec,
    }
    
    print(f"  Average time: {avg_time:.2f}s")
    print(f"  Average tokens: {avg_tokens:.0f}")
    print(f"  Tokens/sec: {tokens_per_sec:.2f}")
    
    del model
    import gc
    gc.collect()

In [ ]:
# Create speed comparison table
speed_data = {
    "Format": [],
    "Tokens/sec": [],
    "Speedup": [],
}

baseline_tps = results.get("FP16", {}).get("tokens_per_sec", 1)

for format_name in ["FP16", "INT8", "INT4"]:
    if format_name in results:
        speed_data["Format"].append(format_name)
        speed_data["Tokens/sec"].append(results[format_name]["tokens_per_sec"])
        speedup = results[format_name]["tokens_per_sec"] / baseline_tps
        speed_data["Speedup"].append(f"{speedup:.2f}x" if format_name != "FP16" else "Baseline")

df_speed = pd.DataFrame(speed_data)
print("\nInference Speed Comparison:")
print(df_speed.to_string(index=False))

# Visualize
try:
    fig, ax = plt.subplots(figsize=(8, 4))
    formats = speed_data["Format"]
    tps = speed_data["Tokens/sec"]
    
    bars = ax.bar(formats, tps, color=colors)
    ax.set_ylabel("Tokens per Second")
    ax.set_title("Inference Speed by Quantization Format")
    
    for bar, t in zip(bars, tps):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{t:.1f}", ha="center", va="bottom")
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Visualization error: {e}")

## Evaluate Code Quality
[back to top](#Table-of-contents:)

Compare the quality of generated code across different quantization formats.

In [ ]:
# Code generation test prompts
test_prompts = [
    {
        "prompt": "Write a Python function to check if a string is a palindrome.",
        "expected_keywords": ["def ", "return", "lower()", "[::-1]"],
        "task": "Palindrome Check",
    },
    {
        "prompt": "Implement a binary search algorithm in Python.",
        "expected_keywords": ["def ", "mid", "low", "high", "while"],
        "task": "Binary Search",
    },
    {
        "prompt": "Write a Python function to flatten a nested list.",
        "expected_keywords": ["def ", "isinstance", "extend", "append"],
        "task": "Flatten List",
    },
    {
        "prompt": "Create a simple Python class for a Stack with push, pop, and peek methods.",
        "expected_keywords": ["class ", "def __init__", "def push", "def pop", "def peek"],
        "task": "Stack Class",
    },
    {
        "prompt": "Write a Python function to find the longest common subsequence of two strings.",
        "expected_keywords": ["def ", "len", "dp", "max"],
        "task": "LCS Algorithm",
    },
]

In [ ]:
import re

def evaluate_code_quality(generated_code, expected_keywords):
    """Evaluate code quality based on keyword matching and structure."""
    score = 0
    details = []
    
    # Check for keywords
    keywords_found = sum(1 for kw in expected_keywords if kw in generated_code)
    keyword_score = keywords_found / len(expected_keywords) * 40
    score += keyword_score
    details.append(f"Keywords: {keywords_found}/{len(expected_keywords)} ({keyword_score:.0f}%)")
    
    # Check for proper function definition
    has_def = "def " in generated_code
    has_return = "return" in generated_code
    has_docstring = '"""' in generated_code or "'''" in generated_code
    
    structure_score = (has_def * 15) + (has_return * 15) + (has_docstring * 10)
    score += structure_score
    details.append(f"Structure: def={has_def}, return={has_return}, docstring={has_docstring}")
    
    # Check for syntax errors (basic)
    try:
        compile(generated_code, '<string>', 'exec')
        syntax_valid = True
    except SyntaxError:
        syntax_valid = False
    
    syntax_score = 20 if syntax_valid else 0
    score += syntax_score
    details.append(f"Syntax valid: {syntax_valid} ({syntax_score}%)")
    
    return score, details

In [ ]:
# Evaluate all formats
all_results = {}

for format_name, model_dir in [("FP16", fp16_dir), ("INT8", int8_dir), ("INT4", int4_dir)]:
    if not model_dir.exists():
        print(f"Skipping {format_name} - directory not found")
        continue
    
    print(f"\n{'='*60}")
    print(f"Evaluating {format_name}...")
    print(f"{'='*60}")
    
    # Load model
    model = OVQwen2_5CoderForCausalLM(model_dir=model_dir, device="CPU", config=config)
    
    format_results = []
    
    for test in test_prompts:
        messages = [{"role": "user", "content": test["prompt"]}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
        
        # Generate
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.8,
            do_sample=True,
        )
        
        generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        # Extract code block if present
        code_match = re.search(r'```python\n(.*?)```', generated, re.DOTALL)
        if code_match:
            code = code_match.group(1).strip()
        else:
            code = generated.strip()
        
        # Evaluate
        score, details = evaluate_code_quality(code, test["expected_keywords"])
        
        format_results.append({
            "task": test["task"],
            "score": score,
            "details": details,
            "generated": generated[:200] + "..." if len(generated) > 200 else generated,
        })
        
        print(f"\nTask: {test['task']}")
        print(f"Score: {score}/100")
        for d in details:
            print(f"  {d}")
    
    all_results[format_name] = format_results
    
    del model
    gc.collect()

In [ ]:
# Create quality comparison table
quality_data = {"Format": [], "Avg Score": [], "Min Score": [], "Max Score": []}

for format_name in ["FP16", "INT8", "INT4"]:
    if format_name in all_results:
        scores = [r["score"] for r in all_results[format_name]]
        quality_data["Format"].append(format_name)
        quality_data["Avg Score"].append(np.mean(scores))
        quality_data["Min Score"].append(np.min(scores))
        quality_data["Max Score"].append(np.max(scores))

df_quality = pd.DataFrame(quality_data)
print("\nCode Quality Comparison:")
print(df_quality.to_string(index=False))

# Visualize
try:
    fig, ax = plt.subplots(figsize=(8, 4))
    formats = quality_data["Format"]
    avg_scores = quality_data["Avg Score"]
    
    bars = ax.bar(formats, avg_scores, color=colors)
    ax.set_ylabel("Average Quality Score")
    ax.set_title("Code Quality by Quantization Format")
    ax.set_ylim(0, 100)
    
    for bar, score in zip(bars, avg_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{score:.1f}", ha="center", va="bottom")
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Visualization error: {e}")

## Summary and Recommendation
[back to top](#Table-of-contents:)

### Final Comparison Table

Based on our testing, here's the complete comparison:

In [ ]:
# Create comprehensive comparison
summary_data = []

for format_name in ["FP16", "INT8", "INT4"]:
    row = {"Format": format_name}
    
    # Size
    if format_name == "FP16":
        row["Size (GB)"] = f"{fp16_size:.2f}"
    elif format_name == "INT8":
        row["Size (GB)"] = f"{int8_size:.2f}"
    else:
        row["Size (GB)"] = f"{int4_size:.2f}"
    
    # Speed
    if format_name in results:
        row["Tokens/sec"] = f"{results[format_name]['tokens_per_sec']:.1f}"
    else:
        row["Tokens/sec"] = "N/A"
    
    # Quality
    if format_name in all_results:
        scores = [r["score"] for r in all_results[format_name]]
        row["Quality Score"] = f"{np.mean(scores):.1f}/100"
    else:
        row["Quality Score"] = "N/A"
    
    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)
print("\n" + "=" * 60)
print("FINAL COMPARISON SUMMARY")
print("=" * 60)
print(df_summary.to_string(index=False))

In [ ]:
# Recommendation
print("\n" + "=" * 60)
print("RECOMMENDATION")
print("=" * 60)
print("""
For most users with 16GB RAM + 8GB GPU:

1. INT4 (RECOMMENDED)
   - Best balance of size, speed, and quality
   - Fits comfortably in 8GB GPU VRAM
   - ~2x faster than FP16
   - Minor quality loss (usually <5%)

2. INT8 (Alternative)
   - Better quality than INT4
   - Requires ~8GB VRAM
   - ~1.5x faster than FP16
   - Good if you have slightly more memory

3. FP16 (Not Recommended for 16GB systems)
   - Best quality but requires ~16GB VRAM
   - Only for systems with 32GB+ RAM
   - Slowest inference speed

VERDICT: Use INT4 for your hardware configuration.
The code quality difference is minimal for most coding tasks.
""")

In [ ]:
# Create comparison visualization
try:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Size comparison
    sizes = [fp16_size, int8_size, int4_size]
    axes[0].bar(formats, sizes, color=colors)
    axes[0].set_title("Model Size")
    axes[0].set_ylabel("GB")
    
    # Speed comparison
    if results:
        tps_values = [results.get(f, {}).get("tokens_per_sec", 0) for f in formats]
        axes[1].bar(formats, tps_values, color=colors)
        axes[1].set_title("Inference Speed")
        axes[1].set_ylabel("Tokens/sec")
    
    # Quality comparison
    if all_results:
        quality_values = []
        for f in formats:
            if f in all_results:
                scores = [r["score"] for r in all_results[f]]
                quality_values.append(np.mean(scores))
            else:
                quality_values.append(0)
        axes[2].bar(formats, quality_values, color=colors)
        axes[2].set_title("Code Quality")
        axes[2].set_ylabel("Score (0-100)")
        axes[2].set_ylim(0, 100)
    
    plt.suptitle("Qwen2.5-Coder Quantization Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Visualization error: {e}")